# OPS gene panel design

This notebook has five top-level sections:
- Load packages and data
- Print data statistics
- Compute best N sets
- Merge best N sets from different datasets
- Compile gene panel

## Load packages and data

### Load packages

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import plotly.express as px
import nbformat
from pathlib import Path
import json
import re
from collections import defaultdict
import numpy as np
from scipy.spatial import ConvexHull, QhullError, distance_matrix
import itertools
import random
#from tqdm.notebook import tqdm
from tqdm import tqdm
import time
from goatools.obo_parser import GODag
from goatools.base import download_ncbi_associations
from goatools.anno.genetogo_reader import Gene2GoReader
from goatools.test_data.genes_NCBI_9606_ProteinCoding import GENEID2NT as GeneID2nt_human
from goatools.base import download_go_basic_obo
from sklearn.cluster import DBSCAN

# Seed the RNG used for gene sampling (random.sample in the best-N selection)
# so the gene panel is reproducible across runs.
random.seed(0)

### Import data

In [ ]:
# **Before public release**: this is the only cell that needs to change.
# Inputs live under ../../data/target_gene_selection (a symlink to the curated
# central HPC dataset; see README). On public release the figshare archive
# populates the same data/ layout, so this path stays the same.
from pathlib import Path

DATA = Path("../../data/target_gene_selection")
exp_folder = str(DATA) + "/"   # downstream cells build paths as exp_folder + "<name>"

# Generated tables (gene panels, pathway-coverage CSVs) go to output/, not the dataset.
OUT_DIR = Path("../../output/target_gene_selection")
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_folder = str(OUT_DIR) + "/"

#### Load the protected list

In [ ]:
with open(exp_folder + 'protected_list.txt', 'r') as file:
    lines = file.readlines()
protected = [line.strip() for line in lines]
protected

protected_genes = [i for i in protected if not "’" in i]
protected_patterns = [i for i in protected if "’" in i]

print(f"length of protected genes: {len(protected_genes)}")
print(f"length of protected patterns: {len(protected_patterns)}")

In [ ]:
protected_genes[0:8]

In [ ]:
protected_patterns = [i[0:-2] for i in protected_patterns]
protected_patterns

#### Load the *A. Funk PhateMapping study*

In [ ]:
#Funk PhateMapping study
funk = pd.read_csv(exp_folder +'Funk_PhateMapping/Funk_Cheeseman_Phate_Data.csv',index_col=0)
print(f"number of columns in Funk PhateMapping study: {len(funk.columns)}")
print(f"dimensionality reduction columns:{[i for i in funk.columns if 'dimensionality_reduction' in i] }")

In [ ]:
funk_subset = funk.copy().loc[funk.interphase_dimensionality_reduction_x<20] #remove genes: 
colnames = ['interphase_dimensionality_reduction_x','interphase_dimensionality_reduction_y']
print("I removed two genes",list(funk.loc[funk.interphase_dimensionality_reduction_x>20].index),"because they were off the chart")

In [ ]:
funk_subset['interphase_dimensionality_reduction_y_flipped']= -funk_subset.copy().interphase_dimensionality_reduction_y
funk_subset.interphase_cluster = funk_subset.interphase_cluster.astype(str)

color_scale = px.colors.qualitative.Dark24 + px.colors.qualitative.Light24 + px.colors.qualitative.Alphabet
fig = px.scatter(x=funk_subset.interphase_dimensionality_reduction_x,
                 y=funk_subset.interphase_dimensionality_reduction_y_flipped,
                 color=funk_subset.interphase_cluster,
                 color_discrete_sequence=color_scale,
                hover_data=[funk_subset.index])
fig.update_layout(
    autosize=False,
    width=800,
    height=800,
)

fig.show()

In [ ]:
funk_subset['mitotic_dimensionality_reduction_y_flipped']= -funk_subset.copy().mitotic_dimensionality_reduction_y
funk_subset.mitotic_cluster = funk_subset.mitotic_cluster.astype(str)
fig = px.scatter(x=funk_subset.mitotic_dimensionality_reduction_x,
                 y=funk_subset.mitotic_dimensionality_reduction_y_flipped,
                 log_x = True,
                 color=funk_subset.mitotic_cluster,
                hover_data=[funk_subset.index])
fig.update_layout(
    autosize=False,
    width=800,
    height=800,
)

fig.show()

In [ ]:
funk = funk_subset.copy().loc[:,['interphase_dimensionality_reduction_x','interphase_dimensionality_reduction_y_flipped',
                                    'mitotic_dimensionality_reduction_x','mitotic_dimensionality_reduction_y',
                                    'interphase_cluster', 'mitotic_cluster'
                                    ]]
# remove nontargeting
funk = funk.loc[~funk.index.str.contains('nontargeting')]

The Funk data is ready to use as `funk`


#### Load the *B. Ramezani PERISCOPE study*

In [ ]:
# Opening JSON file
f = open(exp_folder +'Ramezani_PERISCOPE/A549_umap_frozen_version.json')
 # returns JSON object as 
# a dictionary
data = json.load(f)
f.close()

In [ ]:
print(data.keys())

In [ ]:
#convert the dictionary into a dataframe
ramezani = pd.DataFrame()
ramezani['A549_gene_names'] = list(data['A549_gene_names'])
ramezani['A549_clusterable_embedding_x'] = [i[0] for i in list(data['A549_clusterable_embedding_x'])]
ramezani['A549_clusterable_embedding_y'] = [i[0] for i in list(data['A549_clusterable_embedding_y'])]
ramezani['A549_highlight_labels'] = list(data['A549_highlight_labels'])
ramezani['A549_overall_labels'] = list(data['A549_overall_labels'])
ramezani['A549_other_labels'] = list(data['A549_other_labels'])


In [ ]:
ramezani['A549_highlight_labels'] = ramezani['A549_highlight_labels'].astype(str)
ramezani['A549_overall_labels'] = ramezani['A549_overall_labels'].astype(str)
ramezani['A549_other_labels'] = ramezani['A549_other_labels'].astype(str)
ramezani.set_index('A549_gene_names',inplace=True)

In [ ]:
#print a graph 
fig = px.scatter(x=ramezani['A549_clusterable_embedding_x'],
                 y=ramezani['A549_clusterable_embedding_y'],
                 color=ramezani['A549_highlight_labels'],
                hover_data=[ramezani.index])

fig.update_layout(
    autosize=False,
    width=800,
    height=800,
)

fig.show()

The periscope data is ready to use as `ramezani`

#### Load the *Perturb-Seq* data

In [ ]:
#load in the perturbseq data
replogle = pd.read_csv(exp_folder+'Replogle_PerturbSeq/annotated_embedding_coordinates.csv',index_col=0)
replogle['gene_transcript'] = replogle.index.copy()
replogle.set_index('gene',inplace=True)

seen = set()
uniq = []
doubles = []
for gene in replogle.index:
    if gene not in seen:
        seen.add(gene)
    elif gene in seen:
        doubles.append(gene)
replogle.loc[doubles]

#remove 1332, 3039, 4695, 7156 
replogle['guide_number'] = [i.split("_")[0] for i in replogle.gene_transcript]
replogle = replogle.loc[~replogle.guide_number.isin(['1332','3039','4695','7156'])]

print(len(replogle.index)==len(set(replogle.index)))

In [ ]:
rc = pd.read_csv(exp_folder + 'Replogle_PerturbSeq/clustered_mean_gene_expression_figs2-4.csv.gz',index_col=0).iloc[0]
rc = rc[1:len(rc)]


#add the clusters to the data frame
for id in rc.index:
    replogle.loc[replogle.gene_transcript==id,'cluster'] = rc.loc[id]


In [ ]:
#grab the genes
replogle_genes = list(replogle.index)
print(len(replogle_genes))

#Load in data from RPE1 cells
rpe1 = pd.read_csv(exp_folder + 'Replogle_PerturbSeq/Replogle_RPE1_perturbations.csv',index_col=0)
rpe1 = rpe1[['Number of DEGs (anderson-darling)']]
rpe1['gene'] = [i.split("_")[1] for i in rpe1.index]
rpe1['ensembl'] = [i.split("_")[3] for i in rpe1.index]

#make a dictionary from gene to DEG number
rpe1_dict = defaultdict(list)
for gene in rpe1.gene:
    deg = list(rpe1.loc[rpe1.gene==gene,'Number of DEGs (anderson-darling)'])
    rpe1_dict[gene] = max(deg)
    
#Load in data from K562 cells
k562 = pd.read_csv(exp_folder + 'Replogle_PerturbSeq/Replogle_K562_perturbations.csv',index_col=0)
k562 = k562[['Number of DEGs (anderson-darling)']]
k562['gene'] = [i.split("_")[1] for i in k562.index]
k562['ensembl'] = [i.split("_")[3] for i in k562.index]

#make a dictionary from gene to DEG number
k562_dict = defaultdict(list)
for gene in k562.gene:
    deg = list(k562.loc[k562.gene==gene,'Number of DEGs (anderson-darling)'])
    k562_dict[gene] = max(deg)

#add rpe gene DEG number to the spreadsheet
replogle['min_degs']=0
for gene in list(k562_dict.keys()):
    if not gene in list(replogle.index):
        continue
    replogle.loc[gene,'k562_degs'] = k562_dict[gene]
    replogle.loc[gene,'min_degs'] = k562_dict[gene]

for gene in list(rpe1_dict.keys()):
    if not gene in list(replogle.index):
        continue
    replogle.loc[gene,'rpe1_degs'] = rpe1_dict[gene]
    if rpe1_dict[gene]<replogle.loc[gene,'min_degs']:
        replogle.loc[gene,'min_degs'] = rpe1_dict[gene]
    



replogle['deg_corr'] = np.log10(replogle.k562_degs)/np.log10(replogle.rpe1_degs)

fig = px.scatter(x=replogle.k562_degs,y=replogle.deg_corr,hover_data = [replogle.index])
fig.update_layout(
    autosize=False,
    width=800,
    height=800,
)

fig.show()

In [ ]:
replogle['cell_line_correlation'] = 'Correlated'
replogle.loc[replogle.deg_corr<0.8,'cell_line_correlation'] = 'Smaller effect in K562'
replogle.loc[replogle.deg_corr>1.2,'cell_line_correlation'] = 'Smaller effect in RPE1'
replogle.loc[replogle.min_degs<min(replogle.k562_degs),'cell_line_correlation'] = 'Few DEGs overall'
replogle.cell_line_correlation.value_counts()

In [ ]:
#print a graph
fig = px.scatter(x=replogle['x'],
                 y=replogle['y'],
                 color=replogle['cell_line_correlation'],
                 color_discrete_sequence=['green','orange','blue','grey'],
                hover_data=[replogle.index])

fig.update_layout(
    autosize=False,
    width=600,
    height=400,
)

fig.show()

In [ ]:
replogle = replogle[replogle.cell_line_correlation!='Few DEGs overall'].copy()
#print a graph
fig = px.scatter(x=replogle['x'],
                 y=replogle['y'],
                 color=replogle['cluster'],
                 color_discrete_sequence=['green','orange','blue','grey'],
                hover_data=[replogle.index])

fig.update_layout(
    autosize=False,
    width=600,
    height=400,
)

fig.show()

#### Perturb-seq dataset should now be ready to use as *replogle*

#### Load *UPR* genes from Adamson paper

In [ ]:
upr_genes = list(pd.read_csv(exp_folder+'Adamson_UPR/adamson_s1_uprGenes.csv',index_col=0).index)
print(upr_genes[0:5])
upr_genes[1] = 'AMIGO3'
upr_genes[2] = 'GMPPB'
upr_genes = list(set(upr_genes))
upr_genes = [i for i in upr_genes if not pd.isna(i)]
print(f"number of UPR genes in Adamson study: {len(upr_genes)}")

In [ ]:
#check if Chad's list of 36 genes are in the UPR list
genes_to_remove = pd.read_csv(exp_folder + 'gene_panel_2024_07_17_genes_with_insufficient_gRNA_genes.csv')
gtr = list(genes_to_remove['Gene name'])
[i for i in gtr if i in upr_genes]

#### Load the *CORUM* data

In [ ]:
corum = pd.read_table(exp_folder + 'CORUM/CorumhumanComplexes.txt',index_col=0)
corum['gene_list'] = [i.split(';') for i in corum['subunits(Gene name)']]
corum['synonyms_list'] = [re.split(r'[ ;]+', str(i)) for i in corum['subunits(Gene name syn)']]
corum['gene+syn_list'] = corum['gene_list'] + corum['synonyms_list']
# the "gene+syn_list" column is a list of gene names and their synonyms for each complex

In [ ]:
# make a dictionary of gene names as keys and the list of membership complex names as values
gene2complex = defaultdict(list)
for idx, row in corum.iterrows():
    complex = row['ComplexName']
    for gene in row['gene+syn_list']:
        gene2complex[gene].append(row['ComplexName'])
# deduplicate the list of complex names
for gene in gene2complex:
    gene2complex[gene] = list(set(gene2complex[gene]))


In [ ]:
# make a dictionary of complex names as keys and the list of gene names and their synonyms as values
complex2gene = defaultdict(list)
for idx, row in corum.iterrows():
    complex = row['ComplexName']
    for gene in row['gene+syn_list']:
        complex2gene[complex].append(gene)
# deduplicate the list of gene names
for complex in complex2gene:
    complex2gene[complex] = list(set(complex2gene[complex]))

#### Load the *REACT Pathway* data (CTD)

In [ ]:
ctd = pd.read_csv(exp_folder+'CTD_genes_pathways.csv.gz')
smallest_pathway=20

#limit to REACT pathways
ctd = ctd[ctd.PathwayID.str.startswith('REACT')]

# group once (O(N)) instead of scanning the whole frame per pathway/gene (O(N^2))
pathway2gene = defaultdict(list, {
    pathway: genes
    for pathway, genes in ctd.groupby('PathwayName', sort=False)['GeneSymbol'].apply(list).items()
    if len(genes) >= smallest_pathway
})

#subset ctd
ctd = ctd.loc[ctd.PathwayName.isin(list(pathway2gene.keys()))].copy()

gene2pathway = defaultdict(list, ctd.groupby('GeneSymbol', sort=False)['PathwayName'].apply(list).to_dict())

#### Load in *GO ontology* data 

In [ ]:
#start here so you don't need to re-make this every time
df_go = pd.read_csv(exp_folder+'GO_Pathways.csv',index_col=0)

# group once (O(N)) instead of scanning the whole frame per pathway/gene (O(N^2))
pathway2genego = defaultdict(list, {
    pathway: genes
    for pathway, genes in df_go.groupby('GO_Name', sort=False)['Symbol'].apply(list).items()
    if len(genes) >= smallest_pathway
})

df_go = df_go.loc[df_go.GO_Name.isin(list(pathway2genego.keys()))].copy()

gene2pathwaygo = defaultdict(list, df_go.groupby('Symbol', sort=False)['GO_Name'].apply(list).to_dict())

## Print data statistics

Changing this section to iterate through the names so that I don't have to repeat everything


In [ ]:
complexdict = {'all':[]}
REACTdict = {'all':[]}
godict = {'all':[]}
uprdict = {'all':[]}
protectedgenes_dict = {'all':[]}
protectedpatterns_dict = {'all':[]}



for dataset_name in ["funk","ramezani","replogle"]:
    dataset = globals()[dataset_name]
    dataset['corum_complexes'] = [gene2complex.get(i,[]) for i in dataset.index]
    dataset['REACT_pathways'] = [gene2pathway.get(i,[]) for i in dataset.index]
    dataset['go_pathways'] = [gene2pathwaygo.get(i,[]) for i in dataset.index]
    
    complexdict[dataset_name] = dataset.corum_complexes.loc[dataset.corum_complexes.apply(len)>0]
    REACTdict[dataset_name] = dataset.REACT_pathways.loc[dataset.REACT_pathways.apply(len)>0]
    godict[dataset_name] = dataset.go_pathways.loc[dataset.go_pathways.apply(len)>0]
    uprdict[dataset_name] = [i for i in list(dataset.index) if i in upr_genes]
    protectedgenes_dict[dataset_name] = [i for i in list(dataset.index) if i in protected_genes]
    protectedpatterns_dict[dataset_name] = [i for i in list(dataset.index) if any([i.startswith(j) for j in protected_patterns])]
    print(f"number of genes in {dataset_name} study: {len(dataset)}")
    print(f"number of UPR genes in dataset study: {len(uprdict[dataset_name])} out of {len(upr_genes)}")
    print(f"number of genes in {dataset_name} study that are in CORUM: {len(complexdict[dataset_name])}")
    print(f"number of genes in {dataset_name} study that are in the protected list: {len(protectedgenes_dict[dataset_name])}")
    print(f"number of genes in {dataset_name} study that are in the protected pattern list: {len(protectedpatterns_dict[dataset_name])}")
    complexes = list(set([i for _list in list(complexdict[dataset_name]) for i in _list]))
    REACT = list(set([i for _list in list(REACTdict[dataset_name]) for i in _list]))
    go = list(set([i for _list in list(godict[dataset_name]) for i in _list]))
    
    complexdict['all']+=complexes
    REACTdict['all']+=REACT
    godict['all']+=go
    uprdict['all']+=uprdict[dataset_name]
    protectedgenes_dict['all']+=protectedgenes_dict[dataset_name]
    protectedpatterns_dict['all']+=protectedpatterns_dict[dataset_name]
    
    print(f"number of complexes in {dataset_name} study: {len(complexes)} out of {len(complex2gene)}")
    print(f"number of genes in {dataset_name} study that are in a REACT pathway: {len(REACTdict[dataset_name])}")
    print(f"number of REACT pathways in {dataset_name} study: {len(REACT)} out of {len(pathway2gene)}")
    print(f"number of genes in {dataset_name} study that are in a GO pathway: {len(godict[dataset_name])}")
    print(f"number of go pathways in {dataset_name} study: {len(go)} out of {len(pathway2genego)}")
    print(f"")
    print(f"")
    


In [ ]:
all_genes = list(set(list(replogle.index)+list(ramezani.index)+list(funk.index)))

print(f"number of genes in all studies: {len(all_genes)}")
print(f"number of complexes in all studies: {len(set(complexdict['all']))} out of {len(complex2gene)} equals {round(len(set(complexdict['all']))/len(complex2gene)*100)}%")
print(f"number of REACT pathways in all studies: {len(set(REACTdict['all']))} out of {len(pathway2gene)} equals {round(len(set(REACTdict['all']))/len(pathway2gene)*100)}%")
print(f"number of go pathways in all studies: {len(set(godict['all']))} out of {len(pathway2genego)} equals {round(len(set(godict['all']))/len(pathway2genego)*100)}%")
print(f"number of UPR genes in all studies: {len(set(uprdict['all']))} out of {len(upr_genes)} equals {round(len(set(uprdict['all']))/len(upr_genes)*100)}%")
print(f"number of protected genes in all studies: {len(set(protectedgenes_dict['all']))} out of {len(protected_genes)} equals {round(len(set(protectedgenes_dict['all']))/len(protected_genes)*100)}%")
print(f"number of genes with protected patterns in all studies: {len(set(protectedpatterns_dict['all']))} out of {len(protected_patterns)} patterns")
print(f"total number of protected genes: {len(set(protectedgenes_dict['all']+protectedpatterns_dict['all']))}")

NOTE: the Ramezani and Replogle data don't have meaningful clusters

## Compute best N sets

#### Helper functions

In [ ]:
# helper functions of helper functions  ; )
def compute_pairwise_complexes(gene_list):
    ''' Compute a dictionary where the keys are gene names, and the values are lists of genes in the same complex as the key
        gene_list: list of gene names
    '''
    pairwise_dict = defaultdict(list)
    for gene in gene_list:
        complexes = gene2complex.get(gene, [])
        for complex in complexes:
            genes_in_complex = complex2gene[complex]
            pairwise_dict[gene].extend(genes_in_complex)
    return pairwise_dict


def find_redundancy(genelistA, genelistB, pairwise_dict={}): # pairwise_dict speed up this function by precomputing all the gene pairs that are in the same complex
    ''' Find genes in listB that are in complex with at least one gene in listA
        genelistA: list of gene names
        genelistB: list of gene names
        pairwise_dict: a dictionary of gene pairs that are in the same complex (for fast lookup)
    '''
    gene_setA = set(genelistA)

    redundancy_set = set()
    
    if len(pairwise_dict) == 0: # slow option
        for geneB in genelistB:
            complexesB = gene2complex.get(geneB, [])
            for complexB in complexesB:
                genes_in_complexB = complex2gene[complexB]
                #if any(gene in genes_in_complexB for gene in genelistA):
                if gene_setA.intersection(genes_in_complexB):
                    redundancy_set.add(geneB)
                    break
    else: # use the pairwise_dict for fast lookup
        for geneB in genelistB:
            if geneB in pairwise_dict:
                if gene_setA.intersection(pairwise_dict[geneB]):
                    redundancy_set.add(geneB)
                    continue
    return redundancy_set

def find_max_dist_point(arr1, arr2):
    '''Find the point in arr2 that has the maximum distance to any point in arr1
        input: arr1, arr2: numpy array of 2D coords
    '''
    # Compute pairwise distances between each point in arr1 and each point in arr2
    distances = np.linalg.norm(arr1[:, np.newaxis] - arr2, axis=2)
    
    # Find the minimum distance from each point in arr2 to any point in arr1
    min_distances = np.min(distances, axis=0)
    
    # Find the point in arr2 with the maximum of these minimum distances
    max_dist_index = np.argmax(min_distances)
    
    return arr2[max_dist_index]

def find_redundancy_pairs(genelist, pairwise_dict={}):
    ''' Find genes pairs in genelist that are in complex 
        genelist: list of gene names
        pairwise_dict: a dictionary of gene pairs that are in the same complex (for fast lookup)
    '''
    redundancy_set = set()
    
    if len(pairwise_dict) == 0: # slow option
        raise ValueError("This function is not implemented for the slow option")
    else: # use the pairwise_dict for fast lookup
        for geneB in genelist:
            if geneB in pairwise_dict:
                mates = pairwise_dict[geneB]
                for geneA in mates:
                    if geneA in genelist and geneA != geneB:
                        redundancy_set.add(frozenset([geneA,geneB]))
    return redundancy_set

def pick_from_outlier_cluster(selected, unselected, selected_gene_name, unselected_gene_name, pairwise_complex):
    ''' pick genes from outlier cluster. prioritize picking complex pairs
        For the first complex pair, pick the one closest to the cluster centroid. subsequent pairs should have the greatest distance from all selected points
        if no complex pairs are found, return None
        inputs:
            selected: list of selected coordinates
            unselected: list of unselected coordinates
            selected_genename: list of selected gene names
            unselected_genename: list of unselected gene names
    '''
    if all([unselected_gene_name is not None, unselected is not None]):
        assert len(unselected_gene_name) == len(unselected) if unselected is not None else True
    if all([selected_gene_name is not None, selected is not None]):
        assert len(selected_gene_name) == len(selected) if selected is not None else True
    # check if there are complex pairs in unselected
    complex_pairs = find_redundancy_pairs(unselected_gene_name, pairwise_complex)
    selected_empty = selected is None or len(selected) == 0
    if len(complex_pairs) > 0 and selected_empty:  # select the first pair
        # find the pair closest to the centroid of unselected
        centroid = np.mean(unselected, axis=0)
        min_dist_pair = None
        min_dist = np.inf
        for pair in complex_pairs: # for each pair, compute the distance from the centroid, and update the min_dist_pair
            geneA, geneB = pair
            pointA = unselected[unselected_gene_name.index(geneA)]
            pointB = unselected[unselected_gene_name.index(geneB)]
            distA = np.linalg.norm(pointA - centroid)
            distB = np.linalg.norm(pointB - centroid)
            dist = max(distA, distB)
            if dist <= min_dist:
                min_dist = dist
                min_dist_pair = pair
        if min_dist_pair is not None:
            return min_dist_pair
        else:
            print("No min_dist_pair found, some error occurred")
    if len(complex_pairs) > 0 and not selected_empty: # select the pair with the greatest distance from selected points
        # find the pair with the greatest distance from selected points
        max_dist_pair = None
        max_dist = 0
        for pair in complex_pairs: # for each pair, compute the min distance from selected points, and update the max_dist_pair
            geneA, geneB = pair
            if geneA in selected_gene_name and geneB in selected_gene_name: # this should not happen but just in case
                continue
            pointA = unselected[unselected_gene_name.index(geneA)]
            pointB = unselected[unselected_gene_name.index(geneB)]
            distA = np.min(np.linalg.norm(np.array(selected) - pointA, axis=1))
            distB = np.min(np.linalg.norm(np.array(selected) - pointB, axis=1))
            dist = max(distA, distB)
            if dist >= max_dist:
                max_dist = dist
                max_dist_pair = pair
        if max_dist_pair is not None:
            return max_dist_pair
        else:
            print("No max_dist_pair found, some error occurred")
    if len(complex_pairs) == 0: # no complex pairs found
        return None
    raise ValueError("uncaught error") # execution should not reach this point


In [ ]:
# the actual helper functions

# changed: cluster labels should include "nonoutlier" + actual labels of outlier clusters
def select_n_cluster_aware(coords, n, clusters, dense_labels, genenames, integer_clusters = False, pairwise_complex={}, proportional = True, verbose = False):
    ''' Select n points/coordinates from a collection of coords
        Approach: First, find the coordinates that form a convex hull with the maximum area. Then select all hull points
                  Next, we select point from clusteers. 1/2 of points from outlier clusters and 1/2 from non-outlier clusters 
                  For outliers, select the number of points porportional to the cluster size (select more points from larger clusters)
                  genes that are in complex with selected genes (in the same cluster) are prioritized
        inputs:
            coords: numpy array of 2D coordinates
            n: number of coordinates/points to select
            clusters: a list of cluster labels (same length as coords) (expected "nonoutlier" in the label)
            dense_labels: a list of densely overlapping labels (same length as coords)
            genenames: list of all gene names in current dataset
            integer_clusters: whether the cluster labels are integers (default True)
            proportional: whether to select points in each cluster proportionally to the number of points in the cluster (default True)
        return:
            dictionary with keys = cluster label, value = a list of coordinates
            target number to pick per cluster
    '''
    ### input validation and input processing ###
    if n < 1:
        raise ValueError("n must be greater than 0.")
    
    if n >= len(coords) or n >= 1400:
        raise ValueError("n must be smaller than the total number of samples N, and n must be smaller than 1400 (there are 700 outlier genes to support 1:1 picking, therefore max = 1400)")
    
    # check if the cluster labels matches the number of coordinates
    if len(clusters) != len(coords):
        raise ValueError("The number of cluster labels must match the number of coordinates.")
    
    if integer_clusters:
        # for any non-integer n, round down to the nearest integer
        clusters = [float(i) for i in clusters]
        clusters = [int(i) for i in clusters]
    clusters = [str(i) for i in clusters]
    clusters_unique = list(set(clusters))
        
    ### Compute the convex hull of the entire set of points ###
    hull = ConvexHull(coords)
    hull_point_idx = hull.vertices
    hull_points = coords[hull.vertices]
    
    # remove hull_points that are not unique in the dataset, this creates a problem down the road
    unique_points, counts = np.unique(coords, axis=0, return_counts=True)
    duplicate_points = unique_points[counts > 1]
    hull_points = np.array([row for row in hull_points if not any((row == duplicate_point).all() for duplicate_point in duplicate_points)])
    hull_point_names = [genenames[i] for i in hull.vertices]

    # initialize variables to hold results
    balanced_coords = defaultdict(list) # this is the result dictionary, the key is cluster, value is list of coordinates
    cluster_picked_pair_num = defaultdict(int)
    cluster_picked_gene_num = defaultdict(int)
    
    # add hull_points to the balanced_coords collection
    for idx in hull_point_idx:
        _cluster = clusters[idx]
        _coord = coords[idx]
        balanced_coords[_cluster].append(_coord)

    print(f"[n={n}] unique clusters = {len(clusters_unique)}")
    num_gene_outliers = len([i for i in clusters if i != "nonoutlier"]) # number of genes in outlier cluster
    print(f"[n={n}] number of outlier genes = {num_gene_outliers}") if verbose else None
    
    ############################################
    # go through each cluster, and pick genes  #
    ############################################
    sort_key = int if integer_clusters else str
    for cluster in sorted(clusters_unique, key = sort_key):
        # skip cluster if it had only 1 gene
        cluster_size = len([i for i in clusters if i == cluster])
        if cluster_size == 1:
            continue

        # determine the number of genes to pick for current cluster
        if proportional:
            target_num = max(round(n/2*cluster_size/num_gene_outliers),1) # proportional to the cluster size, total is n/2
            if cluster == "nonoutlier": # for nonoutlier cluster, we want to select 1/2 of the points from nonoutlier clusters
                target_num = max(round(n/2), 1)
        else:
            target_num = max(round(n/len(clusters_unique)), 1) # target number of selected genes per cluster

        #print(f"[n={n}] processing cluster {cluster} (total number of clusters = {len(clusters_unique)}), cluster size {cluster_size}, # genes to pick {target_num}") if verbose else None
        
        # get the coordinates and gene names of the points in the cluster
        idx = [idx for idx, val in enumerate(clusters) if val==cluster]
        cluster_coords = coords[idx,:] # get the coordinates of the points in the cluster
        cluster_genenames = [genenames[i] for i in idx] # get the gene names of the points in the cluster

        ##########################
        # the nonoutlier cluster #
        ##########################
        if cluster == "nonoutlier": # for nonoutlier cluster, we want to use the "largest_dist" algorithm
            # pick the first point in the cluster. The first point is the one that is closest to the centroid of the cluster
            centroid = np.mean(cluster_coords, axis=0) #Calculate the centroid of the cluster
            distances = np.linalg.norm(cluster_coords - centroid, axis=1) #Compute the Euclidean distance from each coordinate to the centroid
            closest_index = np.argmin(distances)
            closest_coordinate = cluster_coords[closest_index] # Find the coordinate with the shortest distance to the centroid
            balanced_coords[cluster].append(closest_coordinate) # add the point to the balanced_coords collection
           
            # pick the remaining points in the nonoutlier cluster
            selected = np.array(balanced_coords[cluster])
            cluster_picked_gene_num[cluster] = len(selected)
            # incremental farthest-point sampling: keep the distance from each candidate to its
            # nearest selected point and update it with only the new point each step. This is
            # O(target*N) instead of recomputing the full distance matrix every step (O(target^2*N)),
            # and selects the exact same points (min is associative, argmax tie-break unchanged).
            min_dist = np.min(np.linalg.norm(cluster_coords[:, np.newaxis] - selected, axis=2), axis=1)
            while len(selected) < target_num:
                new_idx = np.argmax(min_dist)
                new_point = cluster_coords[new_idx]
                balanced_coords[cluster].append(new_point)
                selected = np.concatenate((selected, new_point[np.newaxis,:]), axis = 0)
                min_dist = np.minimum(min_dist, np.linalg.norm(cluster_coords - cluster_coords[new_idx], axis=1))
                cluster_picked_gene_num[cluster] += 1
            #print(f"[n={n}] processed cluster {cluster}, cluster size {cluster_size}, # genes to pick {target_num}, # genes picked {cluster_picked_gene_num[cluster]} ") if verbose else None
            continue

        ###########################
        # outlier/finger clusters #
        ###########################
        # remove *intra-cluster* outlier points
        cluster_centroid = np.mean(cluster_coords, axis = 0) # compute cluster centroid
        distances = np.linalg.norm(cluster_coords - cluster_centroid, axis = 1) # compute distances of each point to the centroid
        z_scores = (distances - np.mean(distances)) / np.std(distances) # compute z-scores
        outlier_points = np.array([i for idx, i in enumerate(cluster_coords) if abs(z_scores[idx]) >= 2]) # define outlier points if >2 std from the centroid
        outlier_count = 0
        if len(outlier_points) > 0 and cluster_size > 5: # require at least 5 points in the cluster to remove outliers
            cluster_coords = np.array([i for i in cluster_coords if not i in outlier_points]) # remove the outlier points from the cluster
            updated_idx = [i for i in idx if coords[i] in cluster_coords] # update idx for grabbing gene names
            cluster_genenames = [genenames[i] for i in updated_idx] # update gene names for the cluster
            outlier_count = cluster_size - len(cluster_coords)

        # initialize variables for picking genes    
        iteration = 0
        protected_count = 0
        selected = None
        selected_gene_name = []
        unselected_gene_name = cluster_genenames 
        unselected_gene_name = [i for i in unselected_gene_name if i not in hull_point_names] # substract the selected gene names (which are part of the hull points)

        # enforce picking the protected genes
        protected_genes_in_cluster = []
        for g in unselected_gene_name:
            if g in protected_genes: #or any([g.startswith(pattern.strip("’")[0]) for pattern in protected_patterns]):
                protected_genes_in_cluster.append(g)
                _idx = genenames.index(g)
                balanced_coords[cluster].append(coords[_idx]) # add to the balanced_coords collection
                
                selected_gene_name.append(g) # add to selected gene names
                if selected is None:
                    selected = np.array(coords[_idx])[np.newaxis,:]
                else:
                    selected = np.concatenate((selected, np.array(coords[_idx])[np.newaxis,:]), axis = 0)
                unselected_gene_name = [i for i in unselected_gene_name if i != g]  # remove from unselected gene names
                protected_count += 1

        # for outlier clusters, we first try pick a complex pair using a new function named pick_from_outlier_cluster(), if no complex pairs are found, we pick the point with the maximum distance from the selected points (pick centroid if first iteration)
        # update the unselected points in cluster
        if len(balanced_coords[cluster])==0:
            cluster_remaining_points = [i for i in cluster_coords]
        else:
            cluster_remaining_points = [i for i in cluster_coords if not i in np.array(balanced_coords[cluster])]

        while len(balanced_coords[cluster]) < target_num:
            if iteration == 0 and len(cluster_remaining_points) > 0:
                pair = pick_from_outlier_cluster(selected = selected, unselected= cluster_remaining_points, selected_gene_name = selected_gene_name, unselected_gene_name = unselected_gene_name, pairwise_complex=pairwise_complex)
                #check if the first pair is in the condensed cloud
                
            pair = pick_from_outlier_cluster(selected = selected, unselected= cluster_remaining_points, selected_gene_name = selected_gene_name, unselected_gene_name = unselected_gene_name, pairwise_complex=pairwise_complex)
            if pair is None: # no complex pairs found
                if iteration == 0: # , pick the first point
                    centroid = np.mean(cluster_coords, axis=0) #Calculate the centroid of the cluster
                    distances = np.linalg.norm(cluster_coords - centroid, axis=1) #Compute the Euclidean distance from each coordinate to the centroid
                    closest_index = np.argmin(distances)
                    new_point = cluster_coords[closest_index] # Find the coordinate with the shortest distance to the centroid
                    balanced_coords[cluster].append(new_point) # add the point to the balanced_coords collection
                    new_point = new_point[np.newaxis,:]
                    if selected is None:
                        selected = np.array(new_point)
                    else:
                        selected = np.concatenate((selected, new_point), axis = 0) # add the point to the bookkeeping collection
                else: #pick the point with the maximum distance from the selected points
                    new_point = find_max_dist_point(selected, cluster_remaining_points)
                    balanced_coords[cluster].append(new_point) # add the point to the balanced_coords collection
                    new_point = new_point[np.newaxis,:] # turn into 2D array for np concatenatation
                    selected = np.concatenate((selected, new_point), axis = 0) # add the point to the bookkeeping collection
                # update
                new_point_idx = np.where((coords == new_point).all(axis=1))[0][0]
                selected_gene_name.append(genenames[new_point_idx])
                unselected_gene_name = [i for i in unselected_gene_name if i != genenames[new_point_idx]]
                indices = np.where((cluster_remaining_points == new_point).all(axis=1))[0]
                if indices.size > 0:    
                    cluster_remaining_points = np.delete(cluster_remaining_points, indices[0], axis=0)
                cluster_picked_gene_num[cluster] += 1
                
            else: # complex pair found
                geneA, geneB = pair
                # add the pair to the selection 
                new_point_idxA = np.where(np.array(genenames) == geneA)[0][0]
                new_point_idxB = np.where(np.array(genenames) == geneB)[0][0]
                new_pointA = coords[new_point_idxA] # get the coordinates
                new_pointB = coords[new_point_idxB] # get the coordinates 
                balanced_coords[cluster].append(new_pointA) # add the point to the selection
                balanced_coords[cluster].append(new_pointB) # add the point to the selection

                #update the bookkeeping
                new_pointA_idx = unselected_gene_name.index(geneA) # get the index for np.delete NOTE: here we use retrieve the index by gene name b/c the coordinates are not unique
                new_pointB_idx = unselected_gene_name.index(geneB) # get the index for np.delete NOTE: here we use retrieve the index by gene name b/c the coordinates are not unique
                cluster_remaining_points = np.delete(cluster_remaining_points, [new_pointA_idx, new_pointB_idx], axis=0) # remove the points from the remaining points
                selected_gene_name.append(geneA) #add the gene name to the bookkeeping
                selected_gene_name.append(geneB) #add the gene name to the bookkeeping
                unselected_gene_name = [i for i in unselected_gene_name if i != geneA and i != geneB] # remove the gene names from the unselected list
                if selected is None:
                    selected = np.concatenate((np.array(new_pointA)[np.newaxis,:], np.array(new_pointB)[np.newaxis,:]), axis = 0) # add the points to the bookkeeping
                else:
                    selected = np.concatenate((selected, np.array(new_pointA)[np.newaxis,:], np.array(new_pointB)[np.newaxis,:]), axis = 0) # add the points to the bookkeeping
                cluster_picked_pair_num[cluster] += 1
                cluster_picked_gene_num[cluster] += 2

            if len(cluster_remaining_points) == 0: # stop the loop if all points in the cluster have been selected (and still no satisfying the target number for this cluster)
                continue

            #print(f"- [iter {iteration} finished] cluster size,selected points,cluster remainder {len(cluster_coords)}/{len(balanced_coords[cluster])}/{len(cluster_coords) - len(balanced_coords[cluster])}") if verbose else None
            iteration += 1

        # finished picking genes for the cluster, now we check if all dense patches are represented
        _dense_coords = [i for idx, i in enumerate(coords) if dense_labels[idx] != "not_in_dense_patch" and clusters[idx] == cluster]
        _dense_labels = [dense_labels[idx] for idx, i in enumerate(coords) if dense_labels[idx] != "not_in_dense_patch" and clusters[idx] == cluster]
        assert len(_dense_coords) == len(_dense_labels) 
        _dense_labels_unique = list(set(_dense_labels))
        if len(_dense_coords) > 0: # there are dense patches in the cluster
            for d_label in _dense_labels_unique:
                _dense_coords_in_patch = np.array([i for idx, i in enumerate(coords) if dense_labels[idx] == d_label]) # get all coordinates in the dense patch
                    # check if the dense patch is represented
                balanced_coords_cluster = np.array(balanced_coords[cluster])
                _dense_coords_in_patch_in_selection = _dense_coords_in_patch[np.isin(_dense_coords_in_patch, balanced_coords_cluster)]
                #print(f"[n={n}] {len(_dense_coords_in_patch_in_selection)} selected genes are in the dense patch labelled {d_label} ") if verbose else None
                if len(_dense_coords_in_patch_in_selection) == 0: # the dense patch is not represented
                    random_dense_point = random.sample([i for idx, i in enumerate(coords) if dense_labels[idx] == d_label],1)[0] # randomly pick a gene from the dense patch if not represented
                    #print(f"adding {random_dense_point} to the balanced_coords") if verbose else None
                    balanced_coords[cluster].append(random_dense_point)

        print(f"[n={n}] processed cluster {cluster}, cluster size {cluster_size} (outlier count = {outlier_count}), # genes to pick {target_num}, # genes picked {cluster_picked_gene_num[cluster]} (besides protected count {protected_count}), complex pairs picked {cluster_picked_pair_num[cluster]} (besides protected) ") if verbose else None


    return balanced_coords, target_num

def select_n_largest_dist(coords, n, include = None , verbose = False):
    '''Select n points/coordinates from coolection of coord
       Approach: First find the coordinates that form a convex hull with the maximum area. Then select all the hull points
                 Next, choose points that have the largest distance from any point in the existing selection, until n points are selected
       Inputs:
            coords: numpy array of 2D coordinates
            n: number of coordinates/points to select
            include: numpy array of coordinates to include (in addition to the hull points)
            '''
    ### input validation and input processing ###
    if n < 1:
        raise ValueError("n must be greater than 0.")
    
    if n >= len(coords):
        raise ValueError("n must be smaller than the total number of samples N.")

    if n < len(include):
        raise ValueError("n must be larger than the include list")
    
    # Compute the convex hull of the entire set of points
    hull = ConvexHull(coords)
    hull_points = coords[hull.vertices]

    # if number of hull points + include list >= n, downsample the hull points and return 
    if len(hull_points) + len(include) >= n:
        random_hull_points = np.array(random.sample(list(hull_points), n-len(include)))
        return np.concatenate((include, random_hull_points), axis = 0)
    else: # select points that is furthest from any selected points
        selected = np.concatenate((include, hull_points), axis = 0)

        # incremental farthest-point sampling (O(n*N) instead of O(n^2*N); identical selections)
        min_dist = np.min(np.linalg.norm(coords[:, np.newaxis] - selected, axis=2), axis=1)
        while len(selected) < n:
            new_idx = np.argmax(min_dist)
            new_point = coords[new_idx]
            selected = np.concatenate((selected, new_point[np.newaxis,:]), axis = 0)
            min_dist = np.minimum(min_dist, np.linalg.norm(coords - coords[new_idx], axis=1))

    return selected 



#### Define the range of best-N sets to compute

In [ ]:
# define the range of best-N sets to compute
start = 900 # start with best-200
end = 1300 # end with best-1000
step = 100

#### Funk best N sets

In [ ]:
# compute distances between cluster centroid to NTC centroid

# load the funk data again and get the non-targeting controls
funk_raw = pd.read_csv(exp_folder +'Funk_PhateMapping/Funk_Cheeseman_Phate_Data.csv',index_col=0)
funk_NTC = funk_raw.copy().loc[funk_raw.index.str.contains('nontargeting')]
funk_NTC['interphase_dimensionality_reduction_y_flipped']= -funk_NTC.copy().interphase_dimensionality_reduction_y
# get the coordinates of the NTC 
NTC_coords = np.array(list(zip(funk_NTC["interphase_dimensionality_reduction_x"].to_list(), 
                  funk_NTC["interphase_dimensionality_reduction_y_flipped"].to_list())))
# compute the centroid of the NTCs
NTC_centroid = np.mean(NTC_coords, axis = 0)
# loop through each cluster and compute the distance of cluster centroid to the NTC centroid
centroid_distances = {} # key = cluster, value = distance
for c in funk["interphase_cluster"].unique():
    subset = funk.loc[funk["interphase_cluster"] == c]
    cluster_coords = np.array(list(zip(subset["interphase_dimensionality_reduction_x"].to_list(), 
                                       subset["interphase_dimensionality_reduction_y_flipped"].to_list())))
    cluster_centroid = np.mean(cluster_coords, axis = 0)
    distance = np.linalg.norm(cluster_centroid - NTC_centroid)
    centroid_distances[c] = distance
# plot a histogram of the distances
plt.hist(list(centroid_distances.values()), bins = 75)
plt.title("Distribution of distance of cluster centroid to NTC centroid")
plt.xlabel("Distance")
plt.ylabel("Frequency")
# draw a line at the distance of the NTC centroid to the global centroid
plt.axvline(0.03, color = "red", linestyle = "--")
plt.show()

In [ ]:
# define outlier and non-outlier clusters
outlier_status = {} # key is cluster label, value is True if it is an outlier, False otherwise
threshold = 0.03
for c in centroid_distances:
    if centroid_distances[c] >= threshold:
        outlier_status[c] = True
    else:
        outlier_status[c] = False

# NOTE: generate a new cluster label where nonoutliers are labeled as "nonoutlier" and outliers are labeled with their original cluster label
funk["interphase_new_cluster_label"] = funk["interphase_cluster"].apply(lambda x: "nonoutlier" if not outlier_status[x] else x)
# 
color_scale = px.colors.qualitative.Dark24 + px.colors.qualitative.Light24 + px.colors.qualitative.Alphabet
fig = px.scatter(x=funk.interphase_dimensionality_reduction_x,
                 y=funk.interphase_dimensionality_reduction_y_flipped,
                 color=funk.interphase_new_cluster_label,
                 color_discrete_sequence=color_scale,
                hover_data=[funk.index])
fig.update_layout(
    autosize=False,
    width=800,
    height=800,
)
fig.update_layout(title_text="Funk dataset with non-outlier clusters merged")

fig.show()

print(f"number of outlier clusters: {len(funk['interphase_new_cluster_label'].unique())-1}")
print(f"number of genes in outlier clusters: {len(funk.loc[funk['interphase_new_cluster_label'] != 'nonoutlier'])}")
print(f"number of genes in non-outlier clusters: {len(funk.loc[funk['interphase_new_cluster_label'] == 'nonoutlier'])}")

In [ ]:
## analyze outliers **in** each cluster and plot the distribution of distances to the cluster centroid ##

dists2clustercentroid = {} # key = cluster, value = a list of distances to the cluster centroid
for c in funk["interphase_cluster"].unique():
    subset = funk.loc[funk["interphase_cluster"] == c]
    cluster_coords = np.array(list(zip(subset["interphase_dimensionality_reduction_x"].to_list(), 
                                       subset["interphase_dimensionality_reduction_y_flipped"].to_list())))
    cluster_centroid = np.mean(cluster_coords, axis = 0)
    # compute distances of each point to the centroid
    distances = np.linalg.norm(cluster_coords - cluster_centroid, axis = 1)
    # save the distances    
    dists2clustercentroid[c] = distances

n_clusters = len(funk["interphase_cluster"].unique())
n_clusters = len([i for i in dists2clustercentroid if len(dists2clustercentroid[i]) >= 40])
fig, axes = plt.subplots(n_clusters, 2, figsize=(11, 2.5 * n_clusters))
i = 0
for c in funk["interphase_cluster"].unique():
    if len(dists2clustercentroid[c]) >= 40:
        # Plot the distribution of distances
        axes[i, 0].hist(dists2clustercentroid[c], bins=75)
        axes[i, 0].set_title(f"Cluster {c}: Distribution of distance to cluster centroid")
        axes[i, 0].set_xlabel("Distance")
        axes[i, 0].set_ylabel("Frequency")

        # Compute z-score and plot
        z_scores = (dists2clustercentroid[c] - np.mean(dists2clustercentroid[c])) / np.std(dists2clustercentroid[c])
        axes[i, 1].hist(z_scores, bins=75, color='orange')
        axes[i, 1].set_title(f"Cluster {c}: Distribution of z-scores of distance to cluster centroid")
        axes[i, 1].set_xlabel("Z-score")
        axes[i, 1].set_ylabel("Frequency")
        i += 1

plt.tight_layout()
plt.show()

######################################################
# we define outlier points if abs(z-score) > thres   #
######################################################
# compute a intra-cluster_outlier label
thres = 1.5
for idx, row in funk.iterrows():
    cluster = row["interphase_cluster"]
    coord = np.array([row["interphase_dimensionality_reduction_x"], row["interphase_dimensionality_reduction_y_flipped"]])
    
    subset = funk.loc[funk["interphase_cluster"] == cluster]
    cluster_coords = np.array(list(zip(subset["interphase_dimensionality_reduction_x"].to_list(), 
                                       subset["interphase_dimensionality_reduction_y_flipped"].to_list())))
    cluster_centroid = np.mean(cluster_coords, axis = 0)
    # compute distances of each point to the centroid
    distances = np.linalg.norm(cluster_coords - cluster_centroid, axis = 1)

    distance = np.linalg.norm(coord - cluster_centroid)

    z_score = (distance - np.mean(distances)) / np.std(distances)
    if abs(z_score) >= thres:
        funk.at[idx, "intra_cluster_outlier"] = True
    else:
        funk.at[idx, "intra_cluster_outlier"] = False


In [ ]:
# plot the outlier points
color_scale = px.colors.qualitative.Dark24 + px.colors.qualitative.Light24 + px.colors.qualitative.Alphabet
fig = px.scatter(x=funk.interphase_dimensionality_reduction_x,
                 y=funk.interphase_dimensionality_reduction_y_flipped,
                 color=funk.intra_cluster_outlier,
                 color_discrete_sequence=color_scale,
                hover_data=[funk.index])
fig.update_layout(
    autosize=False,
    width=800,
    height=800,
)
fig.update_layout(title_text="Funk dataset highlighting intra-cluster outliers")

fig.show()

In [ ]:
# identify dense patches of points (in each cluster that are not outliers)
funk_coords = np.array(list(zip(funk["interphase_dimensionality_reduction_x"].to_list(), 
                  funk["interphase_dimensionality_reduction_y_flipped"].to_list())))

for c in funk["interphase_cluster"].unique():
    if outlier_status[c]:
        subset = funk.loc[funk["interphase_cluster"] == c]
        cluster_coords = np.array(list(zip(subset["interphase_dimensionality_reduction_x"].to_list(), 
                                        subset["interphase_dimensionality_reduction_y_flipped"].to_list())))
        # identify dense patches of points using DBSCAN
        eps_value = 0.0015  # maximum distance between two points to be considered in the same neighborhood
        min_samples_value = 4  # minimum number of points required to form a dense region
        dbscan = DBSCAN(eps=eps_value, min_samples=min_samples_value)
        labels = dbscan.fit_predict(cluster_coords)
        # save the labels to the dataframe
        for idx, coord in enumerate(cluster_coords):
            coord_idx = np.where((funk_coords == coord).all(axis=1))[0][0]
            if labels[idx] == -1:
                funk.loc[funk.index[coord_idx], "dense_patch_boolean"] = False
                funk.loc[funk.index[coord_idx], "dense_patch_label"] = f"not_in_dense_patch"
            else:
                funk.loc[funk.index[coord_idx], "dense_patch_boolean"] = True
                funk.loc[funk.index[coord_idx], "dense_patch_label"] = f"{c}_{labels[idx]}"
                

# plot the dense patches

funk["dense_patch_boolean"] = funk["dense_patch_boolean"].apply(lambda x: True if x else False)
funk["dense_patch_label"].fillna("not_in_dense_patch", inplace=True) 
# 
color_scale = ["#c2c2c2"] + px.colors.qualitative.Dark24 + px.colors.qualitative.Alphabet
fig = px.scatter(x=funk.interphase_dimensionality_reduction_x,
                 y=funk.interphase_dimensionality_reduction_y_flipped,
                 color=funk.dense_patch_label,
                 color_discrete_sequence=color_scale,
                hover_data=[funk.index])
fig.update_layout(
    autosize=False,
    width=800,
    height=800,
)
fig.update_layout(title_text="Funk dataset with intra-cluster dense overlapping patches highlighted")

fig.show()

In [ ]:
# collect inputs for computing the best-N sets
funk_coords = np.array(list(zip(funk["interphase_dimensionality_reduction_x"].to_list(), 
                  funk["interphase_dimensionality_reduction_y_flipped"].to_list())))
funk_clusters = funk["interphase_new_cluster_label"].to_list() # NOTE: use the new cluster labels in which nonoutlier cluster labels are replaced with "nonoutlier"
funk_dense_labels = funk["dense_patch_label"].to_list()
funk_genenames = funk.index.to_list()
# precompute the pairwise complex dictionary (for faster lookup)
funk_pairwise_dict = compute_pairwise_complexes(funk_genenames)

In [ ]:
# compute best_n sets
funk_best_n = {}

with tqdm(total = len(range(start, end, step))) as pbar:
    for n in range(start, end, step):
        pbar.set_description(f"computing best n sets, n = {n}")
        pbar.update(1)
        start_time = time.time()

        funk_best_n[n], num_per_cluster = select_n_cluster_aware(coords = funk_coords, n = n, clusters = funk_clusters, dense_labels= funk_dense_labels, 
                                                                 genenames = funk_genenames, pairwise_complex = funk_pairwise_dict, verbose = True)

        # Measure the elapsed time
        elapsed_time = time.time() - start_time
        print(f"compute time for n={n}: {elapsed_time:.2f} seconds")

In [ ]:
# convert coordinates to gene names
funk_best_n_genes = defaultdict(list)
for n,v in funk_best_n.items():
    for cluster, coords in v.items():
        funk_best_n_genes[n] += [funk.index[(funk.interphase_dimensionality_reduction_x == i) & (funk.interphase_dimensionality_reduction_y_flipped == j)].values[0] for i,j in coords]

In [ ]:
# check the number of genes in each best_n set
for n,v in funk_best_n_genes.items():
    print(f"n={n}, number of genes: {len(v)}")

In [ ]:
#how many protected genes were picked? 
len([i for i in funk_best_n_genes[1000] if i in protected_genes]) #69 protected genes out of 197
len([i for i in funk_best_n_genes[1000] if any([i.startswith(j) for j in protected_patterns])]) #110 out of 214 patterned genes
len([i for i in funk_best_n_genes[1000] if i in upr_genes]) #19 upr genes out of 86

### add Ramezani mito hit genes to the protected list
This block of code replaced the ramezani-best N

In [ ]:
# load mito scores
mito_scores = pd.read_csv(exp_folder + 'Ramezani_PERISCOPE/A549_plate_level_median_per_feat_sig_genes_1_FDR_compartment_specific_hits.csv')
mito_scores = mito_scores.sort_values(by="Mito", ascending=False)
mito_scores

In [ ]:
# get top mito hit genes and add to the protected gene list
n_mito_genes = 50
mito_genes = mito_scores["Gene"].to_list()[:n_mito_genes]
protected_genes = list(set(protected_genes + mito_genes))
print(f"number of protected_genes: {len(protected_genes)}")

In [ ]:
#just go ahead and add upr genes to the protected list
protected_genes = list(set(protected_genes + upr_genes))
print(f"number of protected genes: {len(protected_genes)}")

#remove na 
protected_genes = [i for i in protected_genes if not pd.isna(i)]

#### Replogle best N sets

In [ ]:
# collect inputs for computing the best-N sets
replogle_coords = np.array(list(zip(replogle["x"].to_list(), 
                  replogle["y"].to_list())))
replogle_clusters = replogle["cluster"].to_list()
upr_genes_replogle = [i for i in list(replogle.index) if i in upr_genes]

upr_genes_coords = np.array(list(zip(replogle.x.loc[replogle.index.isin(upr_genes_replogle)].to_list(),
                                      replogle.y.loc[replogle.index.isin(upr_genes_replogle)].to_list())))

replogle_genenames = replogle.index.to_list()
# precompute the pairwise complex dictionary (for faster lookup)
replogle_pairwise_dict = compute_pairwise_complexes(replogle_genenames)

In [ ]:
# compute best_n sets
replogle_best_n = {}
if end > replogle.shape[0]:
    end = round(replogle.shape[0],-2)
print(end)
with tqdm(total = len(range(start, end, step))) as pbar:
    for n in range(start, end, step):
        pbar.set_description(f"computing best n sets, n = {n}")
        pbar.update(1)
        start_time = time.time()

        replogle_best_n[n] = select_n_largest_dist(replogle_coords, n = n, include=upr_genes_coords) # can't use cluster-aware version, b/c ramezani data doesn't contain clusters
        # Measure the elapsed time
        elapsed_time = time.time() - start_time
        print(f"compute time for n={n}: {elapsed_time:.2f} seconds")

In [ ]:
# save the best n sets to file
with open(out_folder + 'replogle_best_n.json', 'w') as f:
    json.dump({k: v.tolist() for k, v in replogle_best_n.items()}, f)        
        

In [ ]:
# convert coordinates to gene names
replogle_best_n_genes = {}
for k,v in replogle_best_n.items():
    replogle_best_n_genes[k] = [replogle.index[(replogle.x == i) & (replogle.y == j)].values[0] for i,j in v]
for k,v in replogle_best_n_genes.items():
    print(f"n={k}, number of genes: {len(v)}")

## Merge best N sets from different datasets

In [ ]:
target_panel_size = 1000

In [ ]:
# helper functions

def redundancy(genelist,gene2complexdict):
    ''' Calculate the percentage of the list is devoted to redundancy (proteins in  the same complex)
    genelist: list of gene names
    returns: the percentage of the list that shares at least one complex with another gene
    '''
    # build each gene's complex set once (was rebuilt inside an O(n^2) double loop);
    # a gene is "redundant" iff it shares >=1 complex with a different gene -> same result.
    complex_sets = [set(gene2complexdict.get(i,[])) for i in genelist]
    count = 0
    for i in range(len(genelist)):
        si = complex_sets[i]
        if si and any(si & complex_sets[j] for j in range(len(genelist)) if j != i):
            count += 1
    return count / len(genelist)

In [ ]:
def redundancy_in_cluster(genelist,gene2complexdict):
    ''' Calculate the percentage of the list is devoted to redundancy (proteins in  the same complex in the same cluster)
    genelist: list of gene names
    returns: the percentage of the list that shares at least one complex with another gene
    '''
    # build each gene's complex set once, and look up funk membership / cluster from a dict
    # (the original rebuilt list(funk.index) for every i and j). Counts a gene iff it shares
    # >=1 complex with a different gene in the same funk cluster -> identical result.
    complex_sets = [set(gene2complexdict.get(i,[])) for i in genelist]
    cluster_of = funk['interphase_new_cluster_label'].to_dict()
    count = 0
    for i, gene1 in enumerate(genelist):
        cluster1 = cluster_of.get(gene1)
        si = complex_sets[i]
        if cluster1 is None or not si:
            continue
        if any(cluster_of.get(genelist[j]) == cluster1 and si & complex_sets[j]
               for j in range(len(genelist)) if j != i):
            count += 1
    return count / len(genelist)

In [ ]:
merged_list = []
gene2priority = {}
gene2source = {}
priority = 1 #1s were added in the smallest best_n_genes, goes up by one as you add genes from the next iteration
for n in range(start, end, step):
    to_add = funk_best_n_genes[n] + replogle_best_n_genes[n] # removed ramezani_best_n_genes[n]
    merged_set = set(merged_list)                       # set lookup instead of O(n) list scan per gene
    new_genes = [i for i in to_add if i not in merged_set]
    merged_list = list(set(merged_list + new_genes))
    for gene in new_genes:
        source=[]
        for dataset in ['funk','replogle']:
            if gene in globals()[dataset+'_best_n_genes'][n]:
                source+=[dataset]
        gene2priority[gene] = priority
        gene2source[gene] = source
    priority += 1
    _redundancy = redundancy(merged_list,gene2complex)
    redundancy_by_cluster = redundancy_in_cluster(merged_list,gene2complex)
    print(f"adding 'best {n}' sets to the final list, number of genes: {len(merged_list)}, redundancy: {_redundancy:.2f}, redundancy within clusters: {redundancy_by_cluster:.2f}")
    if _redundancy >= 0.25 and len(merged_list) > target_panel_size:
        break


## Compile gene panel

  The following note is about the outlier/non-outlier label output:  
The Funk outlier information is in the 'interphase_new_cluster_label' column of the funk dataframe.  
All nonoutlier genes have the "nonoutlier" label, and for all outlier genes, the label is the cluster label  

In [ ]:
# helper functions
def get_genes_in_same_complex(gene, list_of_genes, gene2complexdict,complex2genedict):
    ''' Get all genes (that are in a list) that are in the same complex as the given gene
    '''
    res_list = []
    complexes = gene2complexdict.get(gene,[])
    for c in complexes: # get complexes of the current gene
        g = complex2genedict.get(c,[]) # get genes for current complex
        for i in g:
            if i in list_of_genes:
                res_list.append(i)
    res_list = [i for i in res_list if i != gene] # remove self
    res_list = list(set(res_list)) #dedup
    return res_list

def build_gene_panel_df(merged_list):
    gene_panel_df = pd.DataFrame()

    gene_panel_df['Gene name'] = merged_list
    merged_set = set(merged_list)          # O(1) membership for get_genes_in_same_complex below
    gene_panel_df['Priority (smaller is higher)'] = [gene2priority[i] for i in merged_list]
    gene_panel_df['Dataset(s) gene was selected from'] = [gene2source[i] for i in merged_list]
    gene_panel_df["Funk_map_x"] = [funk.loc[i, 'interphase_dimensionality_reduction_x'] if i in funk.index else np.nan for i in merged_list]
    gene_panel_df["Funk_map_y"] = [funk.loc[i, 'interphase_dimensionality_reduction_y_flipped'] if i in funk.index else np.nan for i in merged_list]
    ramezani_index = set(ramezani.index)   # was rebuilt as a list on every iteration (O(n^2))
    gene_panel_df["Ramezani_map_x"] = [ramezani.loc[i, 'A549_clusterable_embedding_x'] if i in ramezani_index else np.nan for i in merged_list]
    gene_panel_df["Ramezani_map_y"] = [ramezani.loc[i, 'A549_clusterable_embedding_y'] if i in ramezani_index else np.nan for i in merged_list]
    gene_panel_df["Replogle_map_x"] = [replogle.loc[i, 'x'] if i in replogle.index else np.nan for i in merged_list]
    gene_panel_df["Replogle_map_y"] = [replogle.loc[i, 'y'] if i in replogle.index else np.nan for i in merged_list]
    
    gene_panel_df['In_corum_complexes'] = [gene2complex.get(i,[]) for i in merged_list]
    gene_panel_df['In_same_complex_with'] = [get_genes_in_same_complex(i,merged_set,gene2complex,complex2gene) for i in merged_list]

    #add info about pathways
    gene_panel_df['In_REACT_pathways'] = [gene2pathway.get(i,[]) for i in merged_list]
    gene_panel_df['In_same_REACT_pathway_with'] = [get_genes_in_same_complex(i,merged_set,gene2pathway,pathway2gene) for i in merged_list]
    gene_panel_df['In_go_pathways'] = [gene2pathwaygo.get(i,[]) for i in merged_list]
    gene_panel_df['In_same_go_pathway_with'] = [get_genes_in_same_complex(i,merged_set,gene2pathwaygo,pathway2genego) for i in merged_list]
    
    #is the gene on the protected list
    protected_set = set(protected_genes)
    protected_patterns_set = set(protectedpatterns_dict['all'])
    gene_panel_df['In_protected_list'] = [True if i in protected_set else False for i in merged_list]
    gene_panel_df['In_protected_patterns'] = [True if i in protected_patterns_set else False for i in merged_list]
    
    funk_newclusterlabel = funk.loc[funk.index.isin(gene_panel_df['Gene name']),'interphase_new_cluster_label']

    # map the funk cluster label by gene name in one pass (was a boolean-mask scan per funk gene)
    gene_panel_df['funk_cluster'] = gene_panel_df['Gene name'].map(funk_newclusterlabel).fillna('NotInFunk')

    #which dataset(s) is each gene found in? This includes the ramezani dataset and outliers/non-outliers for Funk. 
    # precompute index-membership sets + funk cluster lookup once (was rebuilt per gene -> O(n^2))
    _idx_sets = {'funk': set(funk.index), 'replogle': set(replogle.index), 'ramezani': set(ramezani.index)}
    _funk_clusters = funk['interphase_new_cluster_label']
    datasets_all = []
    for gene in list(gene_panel_df['Gene name']):
        datasets = []
        for dataset in ['funk','replogle','ramezani']:
            if gene in _idx_sets[dataset]:
                d = dataset
                if dataset=='funk':
                    if _funk_clusters.loc[gene]!='nonoutlier':
                        d='funk-outlier'
                    else:
                        d = 'funk-nonoutlier'
                datasets += [d]
        if datasets==[]:
            datasets=['ramezani_only']
        datasets_all.append(datasets)
    
    #add info to the gene panel
    gene_panel_df['datasets_all'] = datasets_all
    #print(gene_panel_df['datasets_all'].value_counts())

    #rename
    gene_panel_df['Gene_Category'] = 'None'
    for i in gene_panel_df.index:
        if gene_panel_df.loc[i,'datasets_all']==['funk-nonoutlier']:
            gene_panel_df.loc[i,'Gene_Category']='Negative_Control'

        elif gene_panel_df.loc[i,'datasets_all'] in [['funk-outlier', 'replogle', 'ramezani'], 
                                                  ['funk-outlier'], 
                                                  ['funk-outlier', 'ramezani'], 
                                                  ['funk-outlier', 'replogle']]:
            gene_panel_df.loc[i,'Gene_Category'] = 'Positive_Control'
        elif gene_panel_df.loc[i,'datasets_all'] in [['funk-nonoutlier', 'replogle'], 
                                                  ['replogle']]:
            gene_panel_df.loc[i,'Gene_Category'] = 'Test_Genes'
    
        elif gene_panel_df.loc[i,'datasets_all'] in [['funk-nonoutlier','replogle','ramezani'],
                                                  ['funk-nonoutlier','ramezani'],
                                                  ['replogle','ramezani']]:
            gene_panel_df.loc[i,'Gene_Category'] = 'Deprioritize'

    
    #currently, the priority for everything is 1. Let's make the priority for everything higher
    gene_panel_df['Priority (smaller is higher)'] = [i+5 for i in gene_panel_df['Priority (smaller is higher)']]
    gene_panel_df['Priority (smaller is higher)'].value_counts()
    
    #In order to get the correct numbers (300 each of test and positive, 150 negative control)
    #Set the priority for negative control to 2
    gene_panel_df.loc[gene_panel_df.Gene_Category=='Negative_Control','Priority (smaller is higher)'] = 2

    #Set the priority for positive controls and test genes to 3
    gene_panel_df.loc[gene_panel_df.Gene_Category.isin(['Test_Genes','Positive_Control']),'Priority (smaller is higher)'] = 3

    #Set the priority for deprioritized genes to 4 [all of these will be thrown out]
    gene_panel_df.loc[gene_panel_df.Gene_Category=='Deprioritize','Priority (smaller is higher)'] = 4
    
    #Set the priority for protected genes to 1
    gene_panel_df.loc[gene_panel_df.In_protected_list,'Priority (smaller is higher)'] = 1
    gene_panel_df.loc[(gene_panel_df.In_protected_list) & (~gene_panel_df.Gene_Category.isin(['Negative_Control','Positive_Control','Test_Genes'])),'Gene_Category'] = 'Protected_Only'
    
    #sort the gene panel by priority
    gene_panel_df.sort_values(by='Priority (smaller is higher)', inplace=True)

    #check
    print(gene_panel_df['Priority (smaller is higher)'].value_counts(sort=False))
    
    return gene_panel_df

In [ ]:
print("building gene panel")
gene_panel_df = build_gene_panel_df(merged_list)
print("The gene panel has",len(gene_panel_df['Gene name']),"genes")

In [ ]:
print(gene_panel_df['Dataset(s) gene was selected from'].value_counts())
print(gene_panel_df['Gene_Category'].value_counts())
print(gene_panel_df['Priority (smaller is higher)'].value_counts(sort=False))

In [ ]:
target_panel_size = 800

In [ ]:
# subset to meet target panel size
# note: the gene panel df should be sorted by priority, so we can remove genes from the bottom of the list
gene_panel_df = gene_panel_df[0:target_panel_size].copy()
print(f"final gene panel has {len(gene_panel_df)} genes")

print(gene_panel_df['Dataset(s) gene was selected from'].value_counts())
print(gene_panel_df['Gene_Category'].value_counts())
print(gene_panel_df['Priority (smaller is higher)'].value_counts(sort=False))

In [ ]:
#check pathway representation - what percent of pathways are represented?
ngenes_REACT_dict = dict()
genenames_REACT_dict = dict()
for pathway in list(set(REACTdict['all'])):
    
    #get all genes associated with the pathway
    p_genes = pathway2gene[pathway]
    
    #subset to genes represented in the study
    p_genes = [i for i in p_genes if i in all_genes]
    
    #get the number of genes per pathway
    total_n_pathway = len(p_genes)
    
    #find out how many of those genes are represented in the gene panel
    p_genes_in_lib = [i for i in p_genes if i in list(gene_panel_df['Gene name'])]
    n_genes_pathway = len(p_genes_in_lib)
    
    #save these numbers to a new dictionary
    ngenes_REACT_dict[pathway] = [total_n_pathway,n_genes_pathway]
    genenames_REACT_dict[pathway] = [p_genes,p_genes_in_lib]

#check go pathway representation - what percent of go pathways are represented?
ngenes_go_dict = dict()
genenames_go_dict = dict()
for pathway in list(set(godict['all'])):
    
    #get all genes associated with the pathway
    p_genes = pathway2genego[pathway]
    
    #subset to genes represented in the study
    p_genes = [i for i in p_genes if i in all_genes]
    
    #get the number of genes per pathway
    total_n_pathway = len(p_genes)
    
    #find out how many of those genes are represented in the gene panel
    p_genes_in_lib = [i for i in p_genes if i in list(gene_panel_df['Gene name'])]
    n_genes_pathway = len(p_genes_in_lib)
    
    #save these numbers to a new dictionary
    ngenes_go_dict[pathway] = [total_n_pathway,n_genes_pathway]
    genenames_go_dict[pathway] = [p_genes,p_genes_in_lib]

    
ngenes_complex_dict = dict()
genenames_complex_dict = dict()
for complex in list(set(complexdict['all'])):
    
    #get all genes associated with the complex
    p_genes = complex2gene[complex]
    
    #subset to genes represented in the study
    p_genes = [i for i in p_genes if i in all_genes]
    
    #get the number of genes per complex
    total_n_complex = len(p_genes)
    
    #find out how many of those genes are represented in the gene panel
    p_genes_in_lib = [i for i in p_genes if i in list(gene_panel_df['Gene name'])]
    n_genes_complex = len(p_genes_in_lib)
    
    #save these numbers to a new dictionary
    ngenes_complex_dict[complex] = [total_n_complex,n_genes_complex]
    genenames_complex_dict[complex] = [p_genes,p_genes_in_lib]
    
#what percent of the complexes are represented in the library
rc = [i[1] for i in ngenes_complex_dict.values() if i[1]>0]
print(f"{len(rc)}, or {round(len(rc)/len(ngenes_complex_dict)*100)}% of complexes are represented in the dictionary out of {len(ngenes_complex_dict)} complexes total")


rp = [i[1] for i in ngenes_REACT_dict.values() if i[1]>0]
print(f"{len(rp)}, or {round(len(rp)/len(ngenes_REACT_dict)*100)}% of REACT pathways are represented in the dictionary out of {len(ngenes_REACT_dict)} pathways total")

rg = [i[1] for i in ngenes_go_dict.values() if i[1]>0]
print(f"{len(rg)}, or {round(len(rg)/len(ngenes_go_dict)*100)}% of go pathways are represented in the dictionary out of {len(ngenes_go_dict)} pathways total")

x = len([i for i in list(set(uprdict['all'])) if i in list(gene_panel_df['Gene name'])])
print(f"{x} or {round(x/len(list(set(uprdict['all'])))*100)}% of UPR genes represented in the panel out of {len(list(set(uprdict['all'])))} found in all datasets")


In [ ]:
#print out a list of missing pathways
missing_react = pd.DataFrame([i for i in ngenes_REACT_dict.keys() if ngenes_REACT_dict[i][1]==0])
missing_go = pd.DataFrame([i for i in ngenes_go_dict.keys() if ngenes_go_dict[i][1]==0])

missing_react.to_csv(out_folder+'REACT_pathways_not_included.csv')
missing_go.to_csv(out_folder+'GO_pathways_not_included.csv')

In [ ]:
#how many genes would you have to add to get to 100% react coverage?
genelist = []
#get the genes that show up the most
for i in list(missing_react[0]):
    genelist+=pathway2gene[i]
    
active_gene_list = list(funk.loc[funk.interphase_new_cluster_label!='nonoutlier'].index)+list(replogle.index)
x = pd.DataFrame(genelist)
x.columns = ['allgenes']
bestgenes = x.allgenes.value_counts()

#start with the best gene, keep adding until you get all of the missing react pathways covered

genes_to_add_react = ['KRAS']
pathways_covered = [i for i in gene2pathway['KRAS'] if i in list(missing_react[0])]

for pathway in list(missing_react[0]):
    if pathway not in pathways_covered:
        #pick the next best gene from that pathway
        genes_to_pick_from = pathway2gene[pathway]
        
        #order based on bestgenes value, then pick the first one, which is the one with the most repeats
        pick_genes = bestgenes.loc[bestgenes.index.isin(genes_to_pick_from)]
        
        #Pick genes that have an effect in one of the screens
        if (len(pick_genes))>1:
            nextgene_list = [i for i in pick_genes.index if i in active_gene_list]
            if len(nextgene_list)==0:
                nextgene = pick_genes.index[0]
            else:
                nextgene = nextgene_list[0]
        else:
            nextgene = pick_genes.index
        
        #add the gene to genes_added
        genes_to_add_react +=[nextgene]
        
        #add the new pathways to pathways_covered
        new_pathways = [i for i in gene2pathway[nextgene] if i in list(missing_react[0])]
        pathways_covered+=new_pathways
        pathways_covered = list(set(pathways_covered))
        
print(f"You would need to add {len(genes_to_add_react)} genes to get 95% REACT coverage")

In [ ]:
#genes to add/spike in - add those from the react list
protected_genes+=genes_to_add_react
protected_genes = list(set(protected_genes))
print(len(protected_genes))
#how many protected genes do we need to add - 225
genes_to_add = [i for i in protected_genes if not i in list(gene_panel_df['Gene name'])]
print(f"There are {len(genes_to_add)} genes on the protected list not found in the gene panel")

In [ ]:
#check coverage of protected patterns
df_sub = gene_panel_df[gene_panel_df.In_protected_patterns]
pgl = list(df_sub['Gene name'])
protected_genes = [i for i in protected_genes if not pd.isna(i)]
pgl += protected_genes
pgl = list(set(pgl))
patterndict = {}
print("Number of genes in all datasets from protected patterns")
for pattern in protected_patterns:
    patterndict[pattern] = [i for i in all_genes if i.startswith(pattern)]
for i,j in patterndict.items():
    print(i,":",len(j))
    
print("Number of genes in gene panel + protected gene list from protected patterns")
for pattern in protected_patterns:
    patterndict[pattern] = [i for i in pgl if i.startswith(pattern)]
for i,j in patterndict.items():
    print(i,":",len(j))

In [ ]:
#potentially add in additional SF3, TOMM, and TIMM genes to the panel (add to protected_genes)

In [ ]:
gene_panel_df.Gene_Category.value_counts()

### Add in genes_to_add and recalculate statistics

In [ ]:
#remake gene panel using current genes + genes_to_add
gene2priority = {}
gene2source = {}
priority = 1
new_genes = list(set(list(gene_panel_df['Gene name'])+genes_to_add))
for gene in new_genes:
    source=[]
    for dataset in ['funk','replogle']:
        if gene in globals()[dataset+'_best_n_genes'][900]:
            source+=[dataset]
    gene2priority[gene] = priority
    gene2source[gene] = source



print("building gene panel with", len(new_genes),"genes")
gene_panel_df = build_gene_panel_df(new_genes)
print("The gene panel has",len(gene_panel_df['Gene name']),"genes")

print(gene_panel_df['Dataset(s) gene was selected from'].value_counts())
print(gene_panel_df['Gene_Category'].value_counts())
print(gene_panel_df['Priority (smaller is higher)'].value_counts(sort=False))

In [ ]:
#load in gene panel from Chad
gene_panel_df = pd.read_csv(exp_folder+'gene_panel_2024_07_17_exclude_insufficient_gRNA_genes.csv',converters = {'In_corum_complexes':pd.eval})
#remove Duo's list
genes_duo = ['CCDC748','UPK3BL1','TMEM8A','KIAA1211','C6orf203','U2AF1','RRP15','ANKRD20A3','DIPK1B','GPALPP1']
gene_panel_df = gene_panel_df.loc[~gene_panel_df['Gene name'].isin(genes_duo)]
gene_panel_df.shape

#Fix Gene Category
gene_panel_df.loc[pd.isna(gene_panel_df['Gene_Category']),'Gene_Category'] = 'Protected_Only'
gene_panel_df.Gene_Category.value_counts()
print(gene_panel_df['Dataset(s) gene was selected from'].value_counts())
print(gene_panel_df['Gene_Category'].value_counts())
print(gene_panel_df['Priority (smaller is higher)'].value_counts(sort=False))

In [ ]:
gene_panel_df.groupby(['In_protected_list','Gene_Category'])['datasets_all'].count()

In [ ]:
gene_panel_df.In_protected_list.value_counts()

In [ ]:
#check pathway representation - what percent of pathways are represented?
ngenes_REACT_dict = dict()
genenames_REACT_dict = dict()
for pathway in list(set(REACTdict['all'])):
    
    #get all genes associated with the pathway
    p_genes = pathway2gene[pathway]
    
    #subset to genes represented in the study
    p_genes = [i for i in p_genes if i in all_genes]
    
    #get the number of genes per pathway
    total_n_pathway = len(p_genes)
    
    #find out how many of those genes are represented in the gene panel
    p_genes_in_lib = [i for i in p_genes if i in list(gene_panel_df['Gene name'])]
    n_genes_pathway = len(p_genes_in_lib)
    
    #save these numbers to a new dictionary
    ngenes_REACT_dict[pathway] = [total_n_pathway,n_genes_pathway]
    genenames_REACT_dict[pathway] = [p_genes,p_genes_in_lib]

#check go pathway representation - what percent of go pathways are represented?
ngenes_go_dict = dict()
genenames_go_dict = dict()
for pathway in list(set(godict['all'])):
    
    #get all genes associated with the pathway
    p_genes = pathway2genego[pathway]
    
    #subset to genes represented in the study
    p_genes = [i for i in p_genes if i in all_genes]
    
    #get the number of genes per pathway
    total_n_pathway = len(p_genes)
    
    #find out how many of those genes are represented in the gene panel
    p_genes_in_lib = [i for i in p_genes if i in list(gene_panel_df['Gene name'])]
    n_genes_pathway = len(p_genes_in_lib)
    
    #save these numbers to a new dictionary
    ngenes_go_dict[pathway] = [total_n_pathway,n_genes_pathway]
    genenames_go_dict[pathway] = [p_genes,p_genes_in_lib]

    
ngenes_complex_dict = dict()
genenames_complex_dict = dict()
for complex in list(set(complexdict['all'])):
    
    #get all genes associated with the complex
    p_genes = complex2gene[complex]
    
    #subset to genes represented in the study
    p_genes = [i for i in p_genes if i in all_genes]
    
    #get the number of genes per complex
    total_n_complex = len(p_genes)
    
    #find out how many of those genes are represented in the gene panel
    p_genes_in_lib = [i for i in p_genes if i in list(gene_panel_df['Gene name'])]
    n_genes_complex = len(p_genes_in_lib)
    
    #save these numbers to a new dictionary
    ngenes_complex_dict[complex] = [total_n_complex,n_genes_complex]
    genenames_complex_dict[complex] = [p_genes,p_genes_in_lib]
    
#what percent of the complexes are represented in the library
rc = [i[1] for i in ngenes_complex_dict.values() if i[1]>0]
print(f"{len(rc)}, or {round(len(rc)/len(ngenes_complex_dict)*100)}% of complexes are represented in the dictionary out of {len(ngenes_complex_dict)} complexes total")

rp = [i[1] for i in ngenes_REACT_dict.values() if i[1]>0]
print(f"{len(rp)}, or {round(len(rp)/len(ngenes_REACT_dict)*100)}% of REACT pathways are represented in the dictionary out of {len(ngenes_REACT_dict)} pathways total")

rg = [i[1] for i in ngenes_go_dict.values() if i[1]>0]
print(f"{len(rg)}, or {round(len(rg)/len(ngenes_go_dict)*100)}% of go pathways are represented in the dictionary out of {len(ngenes_go_dict)} pathways total")

x = len([i for i in upr_genes if i in list(gene_panel_df['Gene name'])])
print(f"{x} or {round(x/len(upr_genes)*100)}% of UPR genes represented in the panel out of {len(upr_genes)}")


In [ ]:
[i for i in upr_genes if not i in list(gene_panel_df['Gene name'])]

In [ ]:
gene_panel_df.shape

In [ ]:
print("computing redundancy")

REACT_red = redundancy(gene_panel_df['Gene name'],gene2pathway)
go_red = redundancy(gene_panel_df['Gene name'],gene2pathwaygo)
c_red = redundancy_in_cluster(gene_panel_df['Gene name'].to_list(),gene2complex)

print(f"complex redundancy within clusters: {c_red:.2f}, \
      redundancy by REACT: {REACT_red:.2f}, \
      redundancy by go: {go_red:.2f}")


In [ ]:
#which GO pathways are missing
x = [list(set(df_go.loc[df_go.GO_Name==i,'Ontology']))[0] for i in ngenes_go_dict.keys() if ngenes_go_dict[i][1]==0]
y = [list(set(df_go.loc[df_go.GO_Name==i,'Ontology']))[0] for i in ngenes_go_dict.keys()]

In [ ]:
#what percent of each GO type are missing from the study
print(len([i for i in x if i=='molecular_function'])/len([i for i in y if i=='molecular_function'])*100)
print(len([i for i in x if i=='biological_process'])/len([i for i in y if i=='biological_process'])*100)
print(len([i for i in x if i=='cellular_component'])/len([i for i in y if i=='cellular_component'])*100)


In [ ]:
#which pathways are under-represented? small pathways
print("REACT pathway size: not present, present")
print(np.median([ngenes_REACT_dict[i][0] for i in list(ngenes_REACT_dict.keys()) if ngenes_REACT_dict[i][1]==0]))
print(np.median([ngenes_REACT_dict[i][0] for i in list(ngenes_REACT_dict.keys()) if ngenes_REACT_dict[i][1]>0]))

#which go pathways are under-represented?
print("Go pathway size: not present, present")
print(np.median([ngenes_go_dict[i][0] for i in list(ngenes_go_dict.keys()) if ngenes_go_dict[i][1]==0]))
print(np.median([ngenes_go_dict[i][0] for i in list(ngenes_go_dict.keys()) if ngenes_go_dict[i][1]>0]))

#which complexes are under-represented? small difference in membership
print("Complex size: not present, present")
print(np.median([ngenes_complex_dict[i][0] for i in list(ngenes_complex_dict.keys()) if ngenes_complex_dict[i][1]==0]))
print(np.median([ngenes_complex_dict[i][0] for i in list(ngenes_complex_dict.keys()) if ngenes_complex_dict[i][1]>0]))

In [ ]:
# save to csv
gene_panel_df.to_csv(out_folder + 'gene_panel_2024-07-18.csv', index=False)

### gene panel visualization and sanity check

In [ ]:
#any columns we need to use - convert to literal rather than string
#gene_panel_df = pd.read_csv(exp_folder+'gene_panel_2024-07-09.csv',converters = {'In_corum_complexes':pd.eval})

also print out the umap plots for each one with the selected genes highlighted




In [ ]:
genes_to_highlight = list(gene_panel_df['Gene name'])
#genes_to_highlight = psm_genes
genes_to_highlight[0:5]

In [ ]:
funk['color'] = 'not_included'
indices = [i for i in list(funk.index) if i in genes_to_highlight]
funk.loc[indices,'color'] = 'included'
#indices = [i for i in list(funk.index) if i in psm_lopri]
#funk.loc[indices,'color'] = 'low_priority'
#indices = [i for i in list(funk.index) if i in psm_nor]
#funk.loc[indices,'color'] = 'no_replogle'
funk.color.value_counts()

fig = px.scatter(x=funk.interphase_dimensionality_reduction_x,
                 y=funk.interphase_dimensionality_reduction_y_flipped,
                 color=funk.color,
                 color_discrete_sequence=['grey','red','blue','green'],
                hover_data=[funk.index,funk.interphase_new_cluster_label])
fig.update_layout(
    autosize=False,
    width=800,
    height=700,
)

fig.show()


In [ ]:
replogle['color'] = 'not_included'
indices = [i for i in list(replogle.index) if i in genes_to_highlight]
replogle.loc[indices,'color'] = 'included'
print(replogle.color.value_counts())

fig = px.scatter(x=replogle['x'],
                 y=replogle['y'],
                 color=replogle['color'],
                 color_discrete_sequence=['grey','red'],
                hover_data=[replogle.index])

fig.update_layout(
    autosize=False,
    width=800,
    height=700,
)

fig.show()

In [ ]:
ramezani['color'] = 'not_included'
indices = [i for i in list(ramezani.index) if i in genes_to_highlight]
ramezani.loc[indices,'color'] = 'included'
print(ramezani.color.value_counts())

fig = px.scatter(x=ramezani['A549_clusterable_embedding_x'],
                 y=ramezani['A549_clusterable_embedding_y'],
                 color=ramezani['color'],
                 color_discrete_sequence=['grey','red'],
                hover_data=[ramezani.index])

fig.update_layout(
    autosize=False,
    width=800,
    height=700,
)

fig.show()

In [ ]:
funk.interphase_new_cluster_label.value_counts()

In [ ]:
#from the datasets: How many of the outlier genes in Funk are represented in our Replogle data? 
funk_outlier_genes = list(funk.loc[funk.interphase_new_cluster_label!='nonoutlier'].index)
funk_nonoutlier_genes = list(funk.loc[funk.interphase_new_cluster_label=='nonoutlier'].index)
funk_replogle_overlap = [i for i in funk_outlier_genes if i in list(replogle.index)]
overlap_in_panel = [i for i in funk_replogle_overlap if i in list(gene_panel_df['Gene name'])]
funk_replogle_nonoverlap = [i for i in funk_nonoutlier_genes if i in list(replogle.index)]
nonoverlap_in_panel = [i for i in funk_replogle_nonoverlap if i in list(gene_panel_df['Gene name'])]

replogle_unique_genes = [i for i in list(replogle.index) if not i in list(funk.index)]
unique_in_panel = [i for i in replogle_unique_genes if i in list(gene_panel_df['Gene name'])]

print(f"{len(funk_replogle_overlap)} genes are overlapping between the funk outliers and the replogle dataset \
out of {len(funk_outlier_genes)} genes in the funk dataset; \
{len(overlap_in_panel)} of these genes are in the panel")
print(f"")
print(f"{len(funk_replogle_nonoverlap)} genes are overlapping between the funk nonoutliers and the replogle dataset \
out of {len(funk_nonoutlier_genes)} genes in the funk dataset; \
{len(nonoverlap_in_panel)} of these genes are in the panel")
print(f"")
print(f"{len(replogle_unique_genes)} genes are present in the replogle dataset but absent from Funk \
out of {len(funk.index)} genes in the funk dataset; \
{len(unique_in_panel)} of these genes are in the panel")

#### find highly redundant complex-clusters
This is how to find genes that are in highly redundant complexes within the same cluster. These could be good candidates for removal. 

In [ ]:
rep_complex_list = []
genedict = {}
max_redundant_genes_per_complex = 5
#find genes that are "in corum complex with" and in the same cluster as other genes
for cluster in list(set(gene_panel_df.funk_cluster)):
    if cluster=='NotInFunk':
        print ('skipping genes not found in clustered dataset')
    else:
        clustergenes = gene_panel_df['Gene name'].loc[gene_panel_df.funk_cluster==cluster]
        clustercomplexes = []
        for i in clustergenes:
            newcomplexes = list(gene_panel_df.loc[gene_panel_df['Gene name'] == i, 'In_corum_complexes'])
            if len(newcomplexes[0])>0:
                clustercomplexes +=newcomplexes[0]
        #how many times is a complex repeated
        df_c = pd.DataFrame(clustercomplexes)
        if df_c.shape[0]!=0:
            df_c.columns = ['Complexes']
            repeated_complexes = df_c.Complexes.value_counts()
            
            #If a complex is represented by > 5 genes, it's put in the "repeated complex" list
            repeated_complexes = repeated_complexes[repeated_complexes>max_redundant_genes_per_complex]
            if len(repeated_complexes.index)>0:
                rep_complex_list+=list(repeated_complexes.index)
                for c in list(repeated_complexes.index):
                    genedict[c] = [i for i in clustergenes if c in gene2complex[i]]
                
rep_complex_list
        

In [ ]:
#how to flatten a list of lists
rep_genes = list(itertools.chain(*[i for i in genedict.values()]))
indices = [list(gene_panel_df.loc[gene_panel_df['Gene name']==i].index)[0] for i in rep_genes]

#make a column with the complex name
complexlist = []
for key in genedict.keys():
    complexlist+=[key]*len(genedict[key])

#Subset the gene panel
df_sub = gene_panel_df.loc[indices]
df_sub.insert(0,'Repeated_Complex',complexlist)
df_sub.to_csv(out_folder+'gene_panel_list_redundant_complexes_in_clusters.csv',index=False)

In [ ]:
#what percent of the genes from each complex are present in the list?
df_complex = pd.DataFrame(ngenes_complex_dict.values())
df_complex.columns = ['Total_Genes','Genes_In_Library']
df_complex.index = ngenes_complex_dict.keys()

fig = px.scatter(x=df_complex.Total_Genes,
                 y=df_complex.Genes_In_Library,
                hover_data=[df_complex.index],
                )

fig.update_layout(
    autosize=False,
    width=600,
    height=400,
    xaxis_title='Total Genes in Complex',
    yaxis_title='Genes in library from Complex'
)

fig.show()

#what percent of the genes from each GO pathway are present in the list? 
df_pathway = pd.DataFrame(ngenes_go_dict.values())
df_pathway.columns = ['Total_Genes','Genes_In_Library']
df_pathway.index = ngenes_go_dict.keys()

fig = px.scatter(x=df_pathway.Total_Genes,
                 y=df_pathway.Genes_In_Library,
                hover_data=[df_pathway.index],
                )

fig.update_layout(
    autosize=False,
    width=600,
    height=400,
    xaxis_title='Total Genes in pathway',
    yaxis_title='Genes in library from pathway'
)

fig.update_xaxes(range=[0,200])
fig.update_yaxes(range=[0,50])

fig.show()

In [ ]:
pd.DataFrame(upr_genes).to_csv(out_folder+'upr_list.csv')